# SALT on Chameleon Cloud (PACT 2026 Reproducibility Challenge)

This notebook provisions the required Chameleon hardware, configures it, launches the released SALT Docker artifact, downloads the results, and releases all resources. Run the cells in order. The normal `smoke` and `reproduce` paths use the checked-in reference PMC data and require no infrastructure beyond the node created here.

**Important:** download every result set you want to keep before running the final teardown cell. The bare-metal node is ephemeral.

## Fixed hardware and image

| Setting | Value |
| --- | --- |
| Site | `CHI@TACC` by default; change one setting to use `CHI@UC` |
| Node type | `compute_skylake` |
| Expected processor | Intel Xeon Gold 6126 (Skylake-SP) |
| Nodes / GPU | 1 bare-metal node / no GPU |
| Image | `CC-Ubuntu24.04` |
| Lease | `smoke`: 4 h; `full`: 16 h (default); `debug`: 48 h; explicitly released during teardown |
| Artifact | Trovi source tree; verified v1.1 archive accepted when present |

The full workflow can take about 8--9 hours after a clean Docker build, which can itself take 1--2 hours. The lease includes safety margin; teardown releases it early.

## 1. Configure the Chameleon project

Select the Chameleon project when prompted. The artifact defaults to `CHI@TACC`; `SITE` is the only setting needed to use `CHI@UC` instead. Hardware type, image, network, and profile durations remain fixed.

In [ ]:
from datetime import datetime, timedelta, timezone
from pathlib import Path
import getpass
import hashlib
import os
import re
import shlex
import tarfile
import time

from chi import context, hardware, lease, server
from chi.exception import ResourceError

SITE = "CHI@TACC"  # Change to "CHI@UC" if that site has the better interval
NODE_TYPE = "compute_skylake"
IMAGE_NAME = "CC-Ubuntu24.04"
RUN_PROFILE = "full"  # "smoke", "full", or developer-only "debug"
LEASE_HOURS = {"smoke": 4, "full": 16, "debug": 48}[RUN_PROFILE]
RELEASE_ARCHIVE = Path("salt-pact26-v1.1.tar.gz").resolve()
RELEASE_SHA256 = "53d8df9da4fc714b86f9fb884f4fcd8b1ed4c0cbec65a401409fd951f5e5b559"
STAGING_ARCHIVE = Path("salt-trovi-source.tar.gz").resolve()
SETUP_SCRIPT = Path("chameleon/setup-node.sh").resolve()
REMOTE_HOME = "/home/cc"
REMOTE_SOURCE = f"{REMOTE_HOME}/salt-artifact"
REMOTE_RESULTS = f"{REMOTE_HOME}/salt-results"
DOCKER_IMAGE = "salt-artifact:v1.1"
SOURCE_PATHS = (
    Path(".cargo"),
    Path(".dockerignore"),
    Path(".envrc.example"),
    Path("Cargo.lock"),
    Path("Cargo.toml"),
    Path("rust-toolchain"),
    Path("analyzer"),
    Path("artifact"),
    Path("benchmarks"),
    Path("cachegrind-runner"),
    Path("denning"),
    Path("raffine"),
    Path("salt_vs_hw_misses_package"),
    Path("scripts"),
    Path("CHANGELOG.md"),
    Path("Dockerfile"),
    Path("LICENSE"),
    Path("README.md"),
    Path("requirements.txt"),
)

raw_owner = os.environ.get("USER") or getpass.getuser()
owner = re.sub(r"[^a-z0-9-]+", "-", raw_owner.lower()).strip("-")
site_tag = SITE.split("@", 1)[-1].lower()
LEASE_NAME = f"{owner}-salt-pact26-{site_tag}-{RUN_PROFILE}"
SERVER_NAME = LEASE_NAME

assert SETUP_SCRIPT.is_file(), f"Missing {SETUP_SCRIPT}"
if RELEASE_ARCHIVE.is_file():
    artifact_source = f"release archive {RELEASE_ARCHIVE.name}"
else:
    missing_paths = [str(path) for path in SOURCE_PATHS if not path.exists()]
    assert not missing_paths, f"Missing Trovi source paths: {missing_paths}"
    artifact_source = "source tree packaged by this notebook"
context.use_site(SITE)
context.choose_project()
print(f"Site: {SITE}")
print(f"Profile: {RUN_PROFILE} ({LEASE_HOURS} hours)")
print(f"Lease/server name: {LEASE_NAME}")
print(f"Artifact source: {artifact_source}")

## 2. Find an available Skylake interval, then reserve it

This cell checks every reservable `compute_skylake` node at `SITE`, chooses the earliest usable host, and reserves that specific node. If the selected profile does not fit immediately, it creates the earliest future reservation and tells you when to resume. Site- and profile-specific names make reruns idempotent without reconnecting to an incompatible lease.

In [ ]:
requested_duration = timedelta(hours=LEASE_HOURS)
SELECTED_SITE = SITE
context.use_site(SITE)
active_leases = [
    item for item in lease.list_leases()
    if item.name == LEASE_NAME
    and str(item.status).upper() in {"PENDING", "ACTIVE"}
]
assert len(active_leases) <= 1, (
    f"Found {len(active_leases)} active leases named {LEASE_NAME} at {SITE}. "
    "Run the cleanup cell before provisioning again."
)

if active_leases:
    l = active_leases[0]
    print(f"Reconnected to existing lease at {SITE}.")
else:
    candidate_nodes = [
        node for node in hardware.get_nodes(node_type=NODE_TYPE) if node.reservable
    ]
    assert candidate_nodes, f"No reservable {NODE_TYPE} nodes are registered at {SITE}"
    availability = []
    for node in candidate_nodes:
        slot_start, slot_end = node.next_free_timeslot(minimum_hours=LEASE_HOURS)
        availability.append((slot_start, node.name, node, slot_end))
    availability.sort(key=lambda item: (item[0], item[1]))

    print(f"Earliest {LEASE_HOURS}-hour {NODE_TYPE} intervals at {SITE} (UTC):")
    for slot_start, node_name, _, slot_end in availability[:10]:
        end_label = slot_end.strftime('%Y-%m-%d %H:%M') if slot_end else 'open-ended'
        print(f"  {node_name}: {slot_start:%Y-%m-%d %H:%M} to {end_label}")

    slot_start, _, selected_node, slot_end = availability[0]
    now = datetime.now(timezone.utc)
    starts_later = slot_start > now + timedelta(minutes=2)
    if starts_later:
        # Use explicit end_date: python-chi computes duration from the current time,
        # not from a supplied future start_date.
        requested_start = slot_start
        requested_end = requested_start + requested_duration
        l = lease.Lease(
            name=LEASE_NAME, start_date=requested_start, end_date=requested_end
        )
    else:
        l = lease.Lease(name=LEASE_NAME, duration=requested_duration)
    l.add_node_reservation(nodes=[selected_node])
    l.add_fip_reservation(amount=1)
    try:
        l.submit(
            wait_for_active=not starts_later,
            wait_timeout=900,
            show="text",
            idempotent=True,
        )
    except ResourceError as exc:
        raise RuntimeError(
            "The availability changed before the reservation was accepted. "
            "Rerun this cell to select the next free node."
        ) from exc

assert l.node_reservations, "The lease has no node reservation"
print(f"Selected site: {SITE}")
print(f"Lease ID: {l.id}; status: {l.status}; starts: {l.start_date}; ends: {l.end_date}")
if l.status != "ACTIVE":
    print(
        "The earliest requested interval has been reserved for the future. "
        "At or after the start time, rerun Sections 1 and 2, then continue to Section 3."
    )

## 3. Launch Ubuntu and wait for SSH

Floating-IP association happens only after Nova reports the server as `ACTIVE`. If a prior attempt left this exact server in `ERROR`, rerunning the cell prints its Nova fault, removes it, and makes one fresh launch attempt within the existing lease.

In [ ]:
l.refresh()
assert l.status == "ACTIVE", (
    f"Lease status is {l.status}; wait until its scheduled start ({l.start_date} UTC), "
    "then rerun Sections 1 and 2."
)

def get_nova_fault(item):
    try:
        sdk_server = item.conn.compute.get_server(item.id)
        return getattr(sdk_server, "fault", None) or {"message": "Nova returned no fault details"}
    except Exception as exc:
        return {"diagnostic_error": str(exc)}

def delete_server_and_wait(item):
    item.delete(idempotent=True, delete_ips=False)
    deadline = time.monotonic() + 300
    while time.monotonic() < deadline:
        if not any(
            candidate.name == SERVER_NAME for candidate in server.list_servers()
        ):
            return
        time.sleep(5)
    raise TimeoutError("Failed server deletion was not confirmed within 5 minutes.")

matching_servers = [
    item for item in server.list_servers() if item.name == SERVER_NAME
]
assert len(matching_servers) <= 1, f"Found multiple servers named {SERVER_NAME}"
s = matching_servers[0] if matching_servers else None

if s is not None:
    s.refresh()
    if s.status == "ERROR":
        previous_fault = get_nova_fault(s)
        print(f"Removing previous ERROR server; Nova fault: {previous_fault}")
        delete_server_and_wait(s)
        s = None
    elif s.status != "ACTIVE":
        s.wait(status="ACTIVE", show="text", timeout=1200)
        s.refresh()

if s is None:
    s = server.Server(
        name=SERVER_NAME,
        reservation_id=l.node_reservations[0]["id"],
        image_name=IMAGE_NAME,
    )
    s.submit(
        wait_for_active=True,
        wait_timeout=1200,
        show="text",
        idempotent=False,
    )
    s.refresh()

if s.status != "ACTIVE":
    fault = get_nova_fault(s)
    raise RuntimeError(
        f"Server provisioning ended in {s.status} at {SITE}. "
        f"Nova fault: {fault}. "
        "Rerun Section 3 once to delete and retry this server; if it repeats, run teardown."
    )

floating_ips = l.get_reserved_floating_ips()
assert floating_ips, "The active lease has no reserved floating IP"
floating_ip = floating_ips[0]
if floating_ip not in s.get_all_floating_ips():
    s.associate_floating_ip(floating_ip)
    s.refresh()
s.check_connectivity(host=floating_ip, timeout=900, show="text")
print(f"Server ID: {s.id}; floating IP: {floating_ip}")

### Recovery after a notebook-kernel restart

If the kernel restarts while the lease is still active, rerun the configuration cell and then this cell. Otherwise continue below.

In [ ]:
context.use_site(SITE)
active_leases = [
    item for item in lease.list_leases()
    if item.name == LEASE_NAME
    and str(item.status).upper() in {"PENDING", "ACTIVE"}
]
assert len(active_leases) == 1, f"Expected one active lease named {LEASE_NAME} at {SITE}; found {len(active_leases)}"
l = active_leases[0]
SELECTED_SITE = SITE
assert l.status == "ACTIVE", f"Lease at {SITE} is {l.status}, not ACTIVE"
s = server.get_server(SERVER_NAME)
s.refresh()
assert s.status == "ACTIVE", f"Server at {SITE} is {s.status}, not ACTIVE"
floating_ip = s.get_floating_ip()
s.check_connectivity(host=floating_ip, timeout=900, show="text")
print(f"Recovered lease {l.id} and server {s.id} at {SITE} / {floating_ip}")

## 4. Define transfer helpers and record the allocated hardware

In [ ]:
def run_remote(command: str):
    return s.execute(command)

def download_tree(remote_directory: str, local_archive: str) -> Path:
    local_path = Path(local_archive).resolve()
    remote_archive = f"{REMOTE_HOME}/{local_path.name}"
    command = (
        f"test -d {shlex.quote(remote_directory)} && "
        f"tar -C {shlex.quote(remote_directory)} -czf {shlex.quote(remote_archive)} ."
    )
    run_remote(command)
    with s.ssh_connection() as connection:
        connection.get(remote_archive, str(local_path))
    print(f"Downloaded {local_path} ({local_path.stat().st_size:,} bytes)")
    return local_path

In [ ]:
hardware_report = run_remote(
    "set -e; uname -a; echo; lscpu; echo; free -h; echo; lsblk; echo; df -h /"
)
Path("chameleon-node-hardware.txt").write_text(
    f"Chameleon site: {SELECTED_SITE}\n\n{hardware_report.stdout}"
)
print("Saved chameleon-node-hardware.txt in the Trovi workspace.")

## 5. Configure the host and stage the artifact source

The setup script installs standard LLVM/MLIR/ChampSim build prerequisites, `perf`, Docker Engine, and the Buildx plugin; enables Docker; adds `cc` to the Docker group; and permits `perf_event_open` on this dedicated leased node. The notebook uses `sudo docker` so it does not depend on group refresh timing.

In [ ]:
s.upload(str(SETUP_SCRIPT), remote_path=f"{REMOTE_HOME}/setup-node.sh")
run_remote(f"chmod 0755 {REMOTE_HOME}/setup-node.sh && sudo {REMOTE_HOME}/setup-node.sh cc")

In [ ]:
def archive_filter(info: tarfile.TarInfo):
    parts = Path(info.name).parts
    if any(part in {"__pycache__", "target", "results"} for part in parts):
        return None
    if info.name.endswith((".pyc", ".pyo")):
        return None
    info.uid = info.gid = 0
    info.uname = info.gname = "root"
    info.mtime = 0
    return info

if RELEASE_ARCHIVE.is_file():
    artifact_archive = RELEASE_ARCHIVE
    local_sha256 = hashlib.sha256(artifact_archive.read_bytes()).hexdigest()
    assert local_sha256 == RELEASE_SHA256, f"Unexpected release archive SHA-256: {local_sha256}"
    archive_description = "verified v1.1 release archive"
else:
    artifact_archive = STAGING_ARCHIVE
    with tarfile.open(artifact_archive, "w:gz") as archive:
        for relative_path in SOURCE_PATHS:
            archive.add(
                relative_path,
                arcname=f"salt-pact26-v1.1/{relative_path.as_posix()}",
                recursive=True,
                filter=archive_filter,
            )
    local_sha256 = hashlib.sha256(artifact_archive.read_bytes()).hexdigest()
    archive_description = "Trovi source-tree staging archive"

remote_archive = f"{REMOTE_HOME}/{artifact_archive.name}"
s.upload(str(artifact_archive), remote_path=remote_archive)
remote_sha256 = run_remote(f"sha256sum {shlex.quote(remote_archive)}").stdout.split()[0]
assert remote_sha256 == local_sha256, (local_sha256, remote_sha256)
run_remote(
    f"set -e; "
    f"if [ ! -f {REMOTE_SOURCE}/.salt-v1.1 ]; then "
    f"test ! -e {REMOTE_SOURCE}; mkdir -p {REMOTE_SOURCE}; "
    f"tar -xzf {shlex.quote(remote_archive)} -C {REMOTE_SOURCE} --strip-components=1; "
    f"touch {REMOTE_SOURCE}/.salt-v1.1; fi"
)
print(f"Staged {archive_description}; SHA-256: {local_sha256}")

## 6. Build and validate the Docker artifact

The first build downloads LLVM/MLIR, Rust, and Python dependencies and may take 1--2 hours. Its toolchain versions are pinned by the artifact's Dockerfile and lock files.

In [ ]:
run_remote(
    f"cd {REMOTE_SOURCE} && "
    f"sudo env DOCKER_BUILDKIT=1 docker build --tag {DOCKER_IMAGE} ."
)

In [ ]:
run_remote(f"sudo docker run --rm --init {DOCKER_IMAGE} test")

## 7A. Recommended first run: end-to-end smoke reproduction

This reduced run exercises all three experiment workflows and should take about 30 seconds once the image exists. It uses the packaged reference PMCs and writes results on the node before downloading them here. Each result path must be new to prevent accidental mixing of runs.

In [ ]:
smoke_results = f"{REMOTE_RESULTS}/smoke"
run_remote(
    f"set -e; test ! -e {smoke_results}; mkdir -p {smoke_results}; "
    f"sudo docker run --rm --init "
    f"--volume {smoke_results}:/artifact/results "
    f"{DOCKER_IMAGE} smoke"
)

In [ ]:
download_tree(smoke_results, "salt-chameleon-smoke-results.tar.gz")

## 7B. Full paper reproduction

Run the next two cells for the complete evaluation. Budget approximately 8--9 hours after the image build. This is independent of the smoke result directory. Keep this browser/kernel session available while the remote command streams output.

In [ ]:
full_results = f"{REMOTE_RESULTS}/full"
assert RUN_PROFILE in {"full", "debug"}, (
    "The full reproduction requires RUN_PROFILE='full' or RUN_PROFILE='debug'."
)

run_remote(
    f"set -e; test ! -e {full_results}; mkdir -p {full_results}; "
    f"sudo docker run --rm --init "
    f"--volume {full_results}:/artifact/results "
    f"{DOCKER_IMAGE} reproduce"
)

In [ ]:
download_tree(full_results, "salt-chameleon-full-results.tar.gz")

## 8. Optional: collect fresh PMCs on the Xeon Gold 6126

This is an additional machine-specific experiment, not the portable Figure 4 reproduction above. The Xeon Gold 6126 is Skylake-SP, whereas the checked-in measurements came from a Core i7-7700 (Kaby Lake). On both validated systems, Linux maps the generic L1D/read/miss selector to `L1D.REPLACEMENT`, not to raw `MEM_LOAD_RETIRED.L1_MISS`. Replacement counts include machine-specific cache-fill and prefetch behavior, so absolute totals can differ. Keep these results separately named.

The first cell disables SMT on this dedicated bare-metal lease and selects an online nonzero logical CPU. For direct host runs, the collector provides the same temporary sibling management and restoration automatically. The second cell grants only the container permissions needed for `perf_event_open`, collects three repeats, and compares those measurements to SALT using the same 32 KiB / 64-byte L1D geometry.

In [ ]:
run_remote(
    "set -e; "
    "if [ -e /sys/devices/system/cpu/smt/control ]; then "
    "echo off | sudo tee /sys/devices/system/cpu/smt/control >/dev/null; fi; "
    "lscpu -e=CPU,CORE,SOCKET,ONLINE"
)
cpu_query = (
    "lscpu -p=CPU,ONLINE | "
    "awk -F, '$1 !~ /^#/ && $2 == \"Y\" && $1 != \"0\" {print $1; exit}'"
)
measurement_cpu = int(run_remote(cpu_query).stdout.strip())
print(f"Fresh measurements will be pinned to logical CPU {measurement_cpu}.")

In [ ]:
pmc_root = f"{REMOTE_RESULTS}/pmc-skylake"
pmc_command = (
    f"python3 salt_vs_hw_misses_package/benchmarks/collect_pmc.py "
    f"--cpu {measurement_cpu} --repeats 3 "
    f"--output /artifact/results/pmc-skylake.csv && "
    f"RESULTS_DIR=/artifact/results/comparison "
    f"./scripts/run-salt-vs-hardware-evaluation.sh "
    f"--pmc /artifact/results/pmc-skylake.csv"
)
run_remote(
    f"set -e; test ! -e {pmc_root}; mkdir -p {pmc_root}; "
    f"sudo docker run --rm --init "
    f"--cap-add PERFMON --security-opt seccomp=unconfined "
    f"--volume {pmc_root}:/artifact/results "
    f"{DOCKER_IMAGE} shell -lc {shlex.quote(pmc_command)}"
)
download_tree(pmc_root, "salt-chameleon-skylake-pmc-results.tar.gz")

## 9. Required teardown

Confirm that the wanted result archives exist in the Jupyter/Trovi file browser, then run this cell. It deletes the server, waits until Nova no longer lists it, and deletes the lease. Deleting the lease releases both the `compute_skylake` node and its reserved floating IP instead of waiting for lease expiry. The cell can also be run after kernel recovery.

In [ ]:
context.use_site(SITE)
matching_servers = [item for item in server.list_servers() if item.name == SERVER_NAME]
for item in matching_servers:
    print(f"Deleting server {item.name} ({item.id}) at {SITE}...")
    item.delete(idempotent=True, delete_ips=False)

deadline = time.monotonic() + 300
while time.monotonic() < deadline:
    if not any(item.name == SERVER_NAME for item in server.list_servers()):
        break
    time.sleep(5)
else:
    raise TimeoutError(f"Server deletion at {SITE} was not confirmed; do not delete the lease yet.")

matching_leases = [
    item for item in lease.list_leases()
    if item.name == LEASE_NAME
    and str(item.status).upper() not in {"TERMINATED", "DELETED"}
]
for item in matching_leases:
    print(f"Deleting lease {item.name} ({item.id}) at {SITE}...")
    item.delete()

if matching_servers or matching_leases:
    print(f"Teardown complete at {SITE}: server, node, and floating IP released.")
else:
    print(f"No matching managed resources remained at {SITE}.")